## Proyecto Medición de neuroblastoma en peces cebra - Kevin Alejandro Gonzalez Villanueva ##
Una de las tareas más importantes que la visión por computador es la de resolver el problema de detección. En este notebook abordaremos la detección de neuroblastoma, un tumor maligno que se forma en el cerebro y que puede ser detectado mediante la visión por computador en un modelo in vivo de peces cebra. Es así que para abordar el problema de detección de neuroblastoma se usarán técnicas de visión por computador como operaciones morfológicas, técnicas de umbralización y segmentación que a posteriori podrán otorgar datos necesarios en la implementación de un entorno que entrene una red neuronal mediante el mecanismo de aprendizaje supervisado. Lo primero que haremos será un análisis de la base de datos y la carga de los mismos. Es importante mencionar que las imágenes están en formato .TIFF.


#### Cargando imágenes

In [ ]:
### Importando librerias
import os
import pandas as pd
import numpy as np
import skimage.io as io
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
import math as math
import tifffile as tiff
import cv2
from skimage.morphology import (binary_erosion, binary_opening, disk, square, diamond)
from skimage.filters import threshold_otsu


In [ ]:

with tiff.TiffFile( 'Imágenes_experimento/REP2_DOXO_01/POST/POZO 6 - TRANSMITIDA.tif') as tif:
    print("n páginas:", len(tif.pages))

In [ ]:
# Cargando las imágenes
# Existen 3 replicas las cuales serán llamadas como R1, R2, R3 respectivamente. Para esto se creó un diccionario que ayuda como directorio de las replicas.
#Cada una de las réplicas tiene un set de imágenes correspondientes as diferentes pozos de agarosa.

REPLICAS = {
    "R1": "REP1_DOXO1",
    "R2": "REP2_DOXO_01",
    "R3": "REP3_CONTROL_ADH"
}

def cargar_imagenes(replica, momento, base_path):
    carpeta = os.path.join(base_path, REPLICAS[replica], momento)
    imagenes = {}

    for archivo in os.listdir(carpeta):
        if not archivo.lower().endswith((".tif", ".tiff")):
            continue

        nombre = os.path.splitext(archivo)[0]
        partes = nombre.upper().split(" - ")

        if len(partes) < 2:
            continue

        pozo = partes[0].replace("POZO", "").strip()
        tipo = partes[1].strip().lower()

        ruta = os.path.join(carpeta, archivo)
        imagenes.setdefault(pozo, {})[tipo] = tiff.imread(ruta)

    return imagenes


Se cargan todos los archivos de las réplicas correspondientes.

In [ ]:
todos = {}
for r in ["R1", "R2", "R3"]:
    todos[r] = {}
    for momento in ["PRE", "POST"]:
        todos[r][momento] = cargar_imagenes(r, momento, base_path="Imágenes_experimento")

        pozos = sorted(todos[r][momento].keys(), key=lambda x: int(x))
    print(f" {r} | {momento} → Pozos: {pozos}")

In [ ]:
def inspeccionar_imagenes(todos, replicas=["R1"], momentos=["PRE", "POST"], max_pozos=3):
    """
    Muestra una grilla: filas = momento (PRE/POST), columnas = pozo
    Para cada pozo muestra TRANSMITIDA arriba y FARRED abajo
    """
    for r in replicas:
        pozos_disponibles = sorted(
            set(todos[r]["PRE"].keys()) & set(todos[r]["POST"].keys()),
            key=lambda x: int(x)
        )[:max_pozos]

        fig, axes = plt.subplots(
            nrows=len(momentos) * 2,   # x2: transmitida + farred
            ncols=len(pozos_disponibles),
            figsize=(5 * len(pozos_disponibles), 4 * len(momentos) * 2)
        )
        fig.suptitle(f"Réplica {r} — Inspección rápida", fontsize=16, fontweight='bold', y=1.01)

        for col, pozo in enumerate(pozos_disponibles):
            for row_m, momento in enumerate(momentos):
                for row_t, tipo in enumerate(["transmitida", "farred"]):
                    ax = axes[row_m * 2 + row_t][col] if len(pozos_disponibles) > 1 else axes[row_m * 2 + row_t]

                    img = todos[r][momento].get(pozo, {}).get(tipo)
                    if img is not None:
                        cmap = 'gray' if tipo == 'transmitida' else 'inferno'
                        ax.imshow(img, cmap=cmap)
                        ax.set_title(f"Pozo {pozo} | {momento}\n{tipo.upper()}", fontsize=9)
                    else:
                        ax.text(0.5, 0.5, "NO DISPONIBLE", ha='center', va='center',
                                transform=ax.transAxes, color='red')

                    ax.axis('off')

        plt.tight_layout()
        plt.show()

# Ejecutar inspección para todas las réplicas, máximo 4 pozos por réplica
inspeccionar_imagenes(todos, replicas=["R1", "R2", "R3"], max_pozos=4)

### Preprocesamiento y umbralización

Antes de aplicar el preprocesamiento y la umbralización, es necesario inspeccionar los histogramas de intensidad de las imágenes para verificar que cumplan el supuesto de bimodalidad. Esta condición es importante porque el algoritmo de Otsu asume un histograma con dos poblaciones bien diferenciadas (fondo y señal) y calcula el umbral en el punto que minimiza la varianza intraclase (o, equivalentemente, maximiza la separación entre ambas clases) en el histograma.

In [ ]:
def visualizar_histograma_farred(todos, replica, momento, pozo, bins=256):

    img = todos[replica][momento][pozo]["farred"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].imshow(img, cmap="inferno")
    axes[0].set_title(f"{replica} | {momento} | Pozo {pozo} | FARRED")
    axes[0].axis("off")

    axes[1].hist(img.ravel(), bins=bins, color="darkred", alpha=0.8)
    axes[1].set_title("Histograma de intensidades")
    axes[1].set_xlabel("Intensidad")
    axes[1].set_ylabel("Frecuencia")

    plt.tight_layout()
    plt.show()
    plt.close()

visualizar_histograma_farred(todos, replica="R3", momento="POST", pozo="7");


Nótese que las frecuencias en el histograma de intensidades son de alta magnitud lo que sugiere en efecto este sí será el histograma correspondiente a la imagen en la izquierda. Nótese además que se debe hacer una reescalación para que el número de intensidades disminuya. Para esto será necesario mencionar que el formato .TIFF es un formato de alta resolución de 16 bits que mostrará niveles de intensidades desde 0 a 65353. Se resuelve esto con normalización lo que mostrará un rango de 0 a 1.

In [ ]:
def normalizar_imagen(img):
    img = np.asarray(img, dtype=np.float32)
    rango = img.max() - img.min()
    imgnorm = (img - img.min()) / rango
    return imgnorm.astype(np.float32, copy=False)

In [ ]:
def otsu_sobre_farred(todos, replica, momento, pozo, mostrar=True):
    """
    Calcula el umbral de Otsu sobre la imagen FARRED y lo dibuja
    sobre el histograma, junto con la máscara binaria resultante.
    """
    img = np.asarray(todos[replica][momento][pozo]["farred"], dtype=np.float32)

    # Normalizar a [0, 1] para que Otsu trabaje estable
    img_norm = normalizar_imagen(img)

    # Umbral de Otsu (en escala 0–1)
    t_otsu = threshold_otsu(img_norm)

    # Máscara binaria
    mascara = img_norm > t_otsu

    if mostrar:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        # Imagen original (normalizada) en gris
        axes[0].imshow(img_norm, cmap="inferno")
        axes[0].set_title(f"{replica} | {momento} | Pozo {pozo}\nFARRED normalizada")
        axes[0].axis("off")

        # Histograma + línea del umbral
        axes[1].hist(img_norm.ravel(), bins=256, range=(0, 1), color="0.3", alpha=0.8)
        axes[1].axvline(t_otsu, color="red", linestyle="--", label=f"Otsu = {t_otsu:.3f}")
        axes[1].set_title("Histograma FARRED (0–1)\n+ umbral de Otsu")
        axes[1].set_xlabel("Intensidad normalizada")
        axes[1].set_ylabel("Frecuencia")
        axes[1].legend()
        # Máscara binaria
        axes[2].imshow(mascara, cmap="gray")
        axes[2].set_title("Máscara binaria (señal tumoral)")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()

    return t_otsu, mascara

In [ ]:
t, mask = otsu_sobre_farred(todos, replica="R3", momento="POST", pozo="7")
print("Umbral Otsu (0–1):", t)

### Análisis a los umbrales de Otsu
Este análisis corresponde a la media y varianza de todos los umbrales calculados aquí.


In [ ]:
# ── Paso 1: calcular y guardar todos los umbrales de Otsu ──────────────────
umbrales_otsu = {}

for replica in todos:
    umbrales_otsu[replica] = {}
    for momento in todos[replica]:
        umbrales_otsu[replica][momento] = {}
        for pozo in todos[replica][momento]:
            t, mascara = otsu_sobre_farred(
                todos, replica, momento, pozo,
                mostrar=False  # no graficar en el loop, solo guardar
            )
            umbrales_otsu[replica][momento][pozo] = t
            print(f"{replica} | {momento} | Pozo {pozo} → Otsu = {t:.4f}")

In [ ]:
print(umbrales_otsu)

In [ ]:
df = pd.read_csv("umbralizaciones.csv")
df = df['Threshold']
print(df)

In [ ]:
df.hist(bins=100); plt.show()


Una vez aplicada la umbralización y habiendo generado la máscara binaria, se encuentra que, primero, el supuesto de bimodalidad se cumple correctamente como se aprecia en el histograma. Ahora se procederá a calcular un umbral que permitirá hacer la máscara binaria correcta para cada una de las imágenes, además; se creará un diccionario que guarde cada una de los valores umbra antes de hacer morfología.

## Morfología y elemento estructurante.

In [ ]:
def analisis_morfologico(todos, replica, momento, pozo,
                         umbrales_otsu,
                         elemento_estructurante=disk(3)):
    """
    Segmentación morfológica:
    1. Umbral de otsu calculado anteriormente
    2. Erosión con elemento estructurante
    3. Apertura con elemento estructurante
    4. Overlay de máscara calculada

    Parámetros:
    -----------
    umbrales_otsu           : dict, umbral de intensidad normalizada OTSU
    elemento_estructurante  : elemento morfológico (disk, square, diamond)
    """
    # ── 1. Cargar y normalizar ──────────────────────────────────────────────
    img = np.asarray(todos[replica][momento][pozo]["farred"], dtype=np.float32)
    img_norm = normalizar_imagen(img)

    # ── 2. Umbral Otsu ──────────────────────────────────────────────────────
    umbral = umbrales_otsu[replica][momento][pozo]
    mascara_umbral = img_norm > umbral

    # ── 3. Erosión ──────────────────────────────────────────────────────────
    mascara_erosion = binary_erosion(mascara_umbral, elemento_estructurante)

    # ── 4. Apertura ─────────────────────────────────────────────────────────
    mascara_apertura = binary_opening(mascara_erosion, elemento_estructurante)

    # ── 5. Visualización ────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(
        f"{replica} | {momento} | Pozo {pozo} — Análisis morfológico (umbral={umbral:.4f})",
        fontsize=12, fontweight="bold"
    )

    titulos = [
        "Original\n(normalizada)",
        f"Umbral > {umbral:.4f}",
        "Erosión",
        "Apertura\n(resultado final)",
        "Overlay\n(apertura sobre original)"
    ]
    imagenes = [
        img_norm,
        mascara_umbral,
        mascara_erosion,
        mascara_apertura,
        None  # overlay
    ]

    for i, (ax, titulo) in enumerate(zip(axes, titulos)):
        if i < 4:
            ax.imshow(imagenes[i], cmap="gray")
        else:
            # Overlay: imagen original + contorno de la máscara final
            ax.imshow(img_norm, cmap="gray")
            ax.contour(mascara_apertura, levels=[0.5], colors="red", linewidths=1)
        ax.set_title(titulo, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    return {
        "mascara_umbral":   mascara_umbral,
        "mascara_erosion":  mascara_erosion,
        "mascara_apertura": mascara_apertura,
        "img_norm":         img_norm
    }

In [ ]:
from skimage.morphology import disk, square, diamond

#SE = disk(5)# círculo de radio 3 px
SE = np.array([
    [0, 1, 0],
    [1, 1, 1],
    [0, 1, 0]
], dtype=np.uint8)
#SE = disk(7)    # círculo más grande
#SE = square(5)
#SE = diamond(7) # diamante de radio 3

In [ ]:
# Prueba en un pozo representativo
resultado = analisis_morfologico(
    todos,
    replica="R2",
    momento="PRE",
    pozo="1",
    umbrales_otsu = umbrales_otsu,
    elemento_estructurante=SE
)

In [ ]:
mascaras = {}

for r in ["R1", "R2", "R3"]:
    mascaras[r] = {}
    for momento in ["PRE", "POST"]:
        mascaras[r][momento] = {}
        for pozo in sorted(todos[r][momento].keys(), key=lambda x: int(x)):
            res = analisis_morfologico(
                todos, replica=r, momento=momento, pozo=pozo,
                umbrales_otsu=umbrales_otsu,
                elemento_estructurante=SE,
            )
            mascaras[r][momento][pozo] = res["mascara_apertura"]

print(" Máscaras morfológicas calculadas para todas las réplicas.")

### Selección del componente tumoral por score e identificación de artefactos de borde

En algunas imágenes la fluorescencia tumoral es débil y el histograma no presenta una bimodalidad clara. En estos casos, la umbralización de Otsu puede incluir ruido de fondo o regiones de autofluorescencia que no corresponden a la señal tumoral.

Por esta razón, en lugar de seleccionar simplemente el componente conexo más grande, se propone seleccionar el componente más probable de corresponder al tumor usando dos criterios:

1. **Eliminación de artefactos de borde:** se descartan componentes que tocan los bordes de la imagen, ya que visualmente se identificó que gran parte del ruido de fondo aparece conectado a los límites de la imagen.
2. **Selección por score:** entre los componentes restantes, se calcula un score basado en el área y la intensidad media del componente:

$score = área \times intensidad\_media$

Este criterio permite favorecer regiones con señal fluorescente más consistente, evitando seleccionar regiones grandes pero tenues que corresponden a ruido.

In [ ]:
from skimage.measure import label, regionprops


def componente_toca_borde(propiedad, shape_imagen, margen_borde=2):
    """
    Determina si un componente conexo toca el borde de la imagen.

    Parámetros
    ----------
    propiedad : RegionProperties
        Propiedades del componente conexo.
    shape_imagen : tuple
        Dimensiones de la imagen binaria: (alto, ancho).
    margen_borde : int, default=0
        Margen adicional para considerar que un componente está cerca del borde.
        Si margen_borde=0, sólo se descartan componentes que tocan exactamente el borde.

    Retorna
    -------
    bool
        True si el componente toca o está dentro del margen del borde.
        False en caso contrario.
    """
    alto, ancho = shape_imagen
    min_fila, min_col, max_fila, max_col = propiedad.bbox

    toca_arriba = min_fila <= margen_borde
    toca_izquierda = min_col <= margen_borde
    toca_abajo = max_fila >= alto - margen_borde
    toca_derecha = max_col >= ancho - margen_borde

    return toca_arriba or toca_izquierda or toca_abajo or toca_derecha


def seleccionar_componente_por_score(
        mascara_apertura,
        img_norm,
        conectividad=2,
        margen_borde=0,
        area_minima=0
):
    """
    Selecciona el componente tumoral más probable desde la máscara de apertura.

    El criterio de selección elimina primero los componentes que tocan el borde
    de la imagen y luego calcula un score para cada componente restante:

        score = área * intensidad_media

    Parámetros
    ----------
    mascara_apertura : ndarray bool
        Máscara binaria resultante de la apertura morfológica.
    img_norm : ndarray float
        Imagen FARRED normalizada en rango [0, 1].
    conectividad : int, default=2
        Conectividad usada para etiquetar componentes.
        En imágenes 2D:
        - 1: vecindad de 4.
        - 2: vecindad de 8.
    margen_borde : int, default=0
        Margen para descartar componentes que tocan o están cerca del borde.
    area_minima : int, default=0
        Área mínima en píxeles para aceptar un componente candidato.

    Retorna
    -------
    mascara_score : ndarray bool
        Máscara binaria del componente seleccionado por score.
    etiquetas : ndarray int
        Imagen etiquetada de componentes conexos.
    df_componentes : DataFrame
        Tabla con métricas de todos los componentes detectados.
    componente_seleccionado : Series o None
        Fila del DataFrame correspondiente al componente seleccionado.
    """
    mascara_apertura = np.asarray(mascara_apertura, dtype=bool)
    img_norm = np.asarray(img_norm, dtype=np.float32)

    etiquetas = label(mascara_apertura, connectivity=conectividad)
    propiedades = regionprops(etiquetas, intensity_image=img_norm)

    registros = []

    for prop in propiedades:
        toca_borde = componente_toca_borde(
            propiedad=prop,
            shape_imagen=mascara_apertura.shape,
            margen_borde=margen_borde
        )

        area = float(prop.area)
        intensidad_media = float(prop.mean_intensity)
        intensidad_total = area * intensidad_media
        score = intensidad_total

        registros.append({
            "Etiqueta": int(prop.label),
            "Area_px": area,
            "Intensidad_media": intensidad_media,
            "Intensidad_total": intensidad_total,
            "Score": score,
            "Toca_borde": toca_borde,
            "BBox": prop.bbox,
            "Centroide": prop.centroid
        })

    df_componentes = pd.DataFrame(registros)

    if df_componentes.empty:
        mascara_score = np.zeros_like(mascara_apertura, dtype=bool)
        return mascara_score, etiquetas, df_componentes, None

    candidatos = df_componentes[
        (~df_componentes["Toca_borde"]) &
        (df_componentes["Area_px"] >= area_minima)
        ].copy()

    if candidatos.empty:
        mascara_score = np.zeros_like(mascara_apertura, dtype=bool)
        return mascara_score, etiquetas, df_componentes, None

    componente_seleccionado = candidatos.sort_values(
        by="Score",
        ascending=False
    ).iloc[0]

    etiqueta_seleccionada = int(componente_seleccionado["Etiqueta"])
    mascara_score = etiquetas == etiqueta_seleccionada

    return mascara_score, etiquetas, df_componentes, componente_seleccionado

In [ ]:
def analisis_morfologico_con_score(
        todos,
        replica,
        momento,
        pozo,
        umbrales_otsu,
        elemento_estructurante=SE,
        conectividad=2,
        margen_borde=0,
        area_minima=0,
        mostrar=True
):
    """
    Aplica segmentación morfológica y selecciona el componente tumoral
    más probable mediante un score basado en área e intensidad.

    Flujo:
    1. Carga de imagen FARRED.
    2. Normalización.
    3. Umbralización con Otsu previamente calculado.
    4. Erosión morfológica.
    5. Apertura morfológica.
    6. Etiquetado de componentes conexos.
    7. Eliminación de componentes que tocan el borde.
    8. Selección del componente con mayor score:

           score = área * intensidad_media

    Parámetros
    ----------
    todos : dict
        Diccionario con todas las imágenes cargadas.
    replica : str
        Identificador de réplica.
    momento : str
        Momento experimental: "PRE" o "POST".
    pozo : str
        Identificador del pozo.
    umbrales_otsu : dict
        Diccionario con umbrales de Otsu calculados previamente.
    elemento_estructurante : ndarray
        Elemento estructurante usado en las operaciones morfológicas.
    conectividad : int, default=2
        Conectividad para etiquetar componentes.
    margen_borde : int, default=0
        Margen para eliminar artefactos de borde.
    area_minima : int, default=0
        Área mínima para aceptar un componente candidato.
    mostrar : bool, default=True
        Si True, muestra las gráficas comparativas.

    Retorna
    -------
    resultado : dict
        Diccionario con las máscaras intermedias, componente seleccionado
        y tabla de componentes.
    """
    # -------------------------------------------------------------------------
    # 1. Cargar imagen FARRED
    # -------------------------------------------------------------------------
    img = np.asarray(
        todos[replica][momento][pozo]["farred"],
        dtype=np.float32
    )

    # -------------------------------------------------------------------------
    # 2. Normalizar imagen
    # -------------------------------------------------------------------------
    img_norm = normalizar_imagen(img)

    # -------------------------------------------------------------------------
    # 3. Umbralización con Otsu
    # -------------------------------------------------------------------------
    umbral = float(umbrales_otsu[replica][momento][pozo])
    mascara_umbral = img_norm > umbral

    # -------------------------------------------------------------------------
    # 4. Erosión morfológica
    # -------------------------------------------------------------------------
    mascara_erosion = binary_erosion(
        mascara_umbral,
        elemento_estructurante
    )

    # -------------------------------------------------------------------------
    # 5. Apertura morfológica
    # -------------------------------------------------------------------------
    mascara_apertura = binary_opening(
        mascara_erosion,
        elemento_estructurante
    )

    # -------------------------------------------------------------------------
    # 6. Selección de componente por score
    # -------------------------------------------------------------------------
    mascara_score, etiquetas, df_componentes, componente_seleccionado = seleccionar_componente_por_score(
        mascara_apertura=mascara_apertura,
        img_norm=img_norm,
        conectividad=conectividad,
        margen_borde=margen_borde,
        area_minima=area_minima
    )

    # -------------------------------------------------------------------------
    # 7. Métricas del componente seleccionado
    # -------------------------------------------------------------------------
    if componente_seleccionado is None:
        etiqueta_score = None
        area_score_px = 0
        intensidad_media_score = 0
        intensidad_total_score = 0
        score_final = 0
        bbox_score = None
        centroide_score = None
    else:
        etiqueta_score = int(componente_seleccionado["Etiqueta"])
        area_score_px = float(componente_seleccionado["Area_px"])
        intensidad_media_score = float(componente_seleccionado["Intensidad_media"])
        intensidad_total_score = float(componente_seleccionado["Intensidad_total"])
        score_final = float(componente_seleccionado["Score"])
        bbox_score = componente_seleccionado["BBox"]
        centroide_score = componente_seleccionado["Centroide"]

    # -------------------------------------------------------------------------
    # 8. Visualización comparativa
    # -------------------------------------------------------------------------
    if mostrar:
        fig, axes = plt.subplots(1, 7, figsize=(28, 4))

        fig.suptitle(
            f"{replica} | {momento} | Pozo {pozo} — "
            f"Segmentación con selección por score\n"
            f"Umbral Otsu = {umbral:.4f} | "
            f"Área = {area_score_px:.0f} px | "
            f"Intensidad media = {intensidad_media_score:.4f} | "
            f"Score = {score_final:.2f}",
            fontsize=13,
            fontweight="bold"
        )

        axes[0].imshow(img_norm, cmap="inferno")
        axes[0].set_title("FARRED\nnormalizada")
        axes[0].axis("off")

        axes[1].imshow(mascara_umbral, cmap="gray")
        axes[1].set_title("Máscara\nOtsu")
        axes[1].axis("off")

        axes[2].imshow(mascara_erosion, cmap="gray")
        axes[2].set_title("Máscara\nerosionada")
        axes[2].axis("off")

        axes[3].imshow(mascara_apertura, cmap="gray")
        axes[3].set_title("Máscara\napertura")
        axes[3].axis("off")

        axes[4].imshow(etiquetas, cmap="nipy_spectral")
        axes[4].set_title("Componentes\nconexos")
        axes[4].axis("off")

        axes[5].imshow(mascara_score, cmap="gray")
        axes[5].set_title("Componente seleccionado\npor score")
        axes[5].axis("off")

        axes[6].imshow(img_norm, cmap="gray")

        if np.any(mascara_apertura):
            axes[6].contour(
                mascara_apertura,
                levels=[0.5],
                colors="yellow",
                linewidths=0.8
            )

        if np.any(mascara_score):
            axes[6].contour(
                mascara_score,
                levels=[0.5],
                colors="red",
                linewidths=1.2
            )

        axes[6].set_title("Overlay\namarillo: apertura\nrojo: score")
        axes[6].axis("off")

        plt.tight_layout()
        plt.show()

    # -------------------------------------------------------------------------
    # 9. Retornar resultados
    # -------------------------------------------------------------------------
    resultado = {
        "img_norm": img_norm,
        "umbral_otsu": umbral,
        "mascara_umbral": mascara_umbral,
        "mascara_erosion": mascara_erosion,
        "mascara_apertura": mascara_apertura,
        "etiquetas_componentes": etiquetas,
        "df_componentes": df_componentes,
        "componente_seleccionado": componente_seleccionado,
        "mascara_score": mascara_score,
        "etiqueta_score": etiqueta_score,
        "area_score_px": area_score_px,
        "intensidad_media_score": intensidad_media_score,
        "intensidad_total_score": intensidad_total_score,
        "score_final": score_final,
        "bbox_score": bbox_score,
        "centroide_score": centroide_score
    }

    return resultado

In [ ]:
resultado_score = analisis_morfologico_con_score(
    todos=todos,
    replica="R3",
    momento="POST",
    pozo="7",
    umbrales_otsu=umbrales_otsu,
    elemento_estructurante=SE,
    conectividad=2,
    margen_borde=0,
    area_minima=0,
    mostrar=True
)

print("Componente seleccionado por score:")
display(resultado_score["componente_seleccionado"])

print("Tabla completa de componentes:")
display(resultado_score["df_componentes"].sort_values("Score", ascending=False).head(10))

In [ ]:
mascaras_score = {}
resultados_score = {}
metricas_score = []

for r in ["R1", "R2", "R3"]:
    mascaras_score[r] = {}
    resultados_score[r] = {}

    for momento in ["PRE", "POST"]:
        mascaras_score[r][momento] = {}
        resultados_score[r][momento] = {}

        pozos_ordenados = sorted(
            todos[r][momento].keys(),
            key=lambda x: int(x)
        )

        for pozo in pozos_ordenados:
            res = analisis_morfologico_con_score(
                todos=todos,
                replica=r,
                momento=momento,
                pozo=pozo,
                umbrales_otsu=umbrales_otsu,
                elemento_estructurante=SE,
                conectividad=2,
                margen_borde=0,
                area_minima=0,
                mostrar=True
            )

            mascaras_score[r][momento][pozo] = res["mascara_score"]
            resultados_score[r][momento][pozo] = res

            metricas_score.append({
                "Replica": r,
                "Momento": momento,
                "Pozo": pozo,
                "Umbral_Otsu": float(res["umbral_otsu"]),
                "Etiqueta_score": res["etiqueta_score"],
                "Area_score_px": float(res["area_score_px"]),
                "Intensidad_media_score": float(res["intensidad_media_score"]),
                "Intensidad_total_score": float(res["intensidad_total_score"]),
                "Score_final": float(res["score_final"]),
                "BBox": res["bbox_score"],
                "Centroide": res["centroide_score"]
            })

df_metricas_score = pd.DataFrame(metricas_score)

print("Proceso finalizado: segmentación por score calculada para todas las imágenes.")
display(df_metricas_score)


### Visualización final: overlay de la máscara tumoral sobre imagen transmitida

Para validar visualmente la segmentación final, se superpone la máscara tumoral seleccionada por score sobre la imagen transmitida correspondiente. Esta visualización permite verificar la localización anatómica de la señal segmentada y confirmar si la máscara final coincide con la región tumoral esperada.

In [ ]:
def cargar_canal_desde_ruta(replica, momento, pozo, canal, base_path="Imágenes_experimento"):
    """
    Carga una imagen TIFF desde disco usando la estructura:

    base_path / carpeta_replica / momento / POZO X - CANAL.tif

    Si el archivo no existe, retorna None.
    """
    ruta = os.path.join(
        base_path,
        REPLICAS[replica],
        momento,
        f"POZO {pozo} - {canal}.tif"
    )

    if not os.path.exists(ruta):
        return None

    return tiff.imread(ruta)


def plot_overlay_farred_transmitida_mascara(
        mascaras_score,
        replica,
        momento,
        pozo,
        base_path="Imágenes_experimento",
        alpha_farred=0.75,
        mostrar_contorno=True,
        color_contorno="cyan"
):
    """
    Grafica la imagen TRANSMITIDA como fondo y superpone la señal FARRED
    con cmap='inferno' únicamente dentro de la máscara de detección final.

    Si falta FARRED o TRANSMITIDA, salta la imagen sin detener el loop.
    """
    img_transmitida = cargar_canal_desde_ruta(
        replica=replica,
        momento=momento,
        pozo=pozo,
        canal="TRANSMITIDA",
        base_path=base_path
    )

    img_farred = cargar_canal_desde_ruta(
        replica=replica,
        momento=momento,
        pozo=pozo,
        canal="FARRED",
        base_path=base_path
    )

    if img_transmitida is None:
        print(f"Saltado: {replica} | {momento} | Pozo {pozo} → no existe TRANSMITIDA.")
        return

    if img_farred is None:
        print(f"Saltado: {replica} | {momento} | Pozo {pozo} → no existe FARRED.")
        return

    mascara_final = np.asarray(
        mascaras_score[replica][momento][pozo],
        dtype=bool
    )

    img_transmitida_norm = normalizar_imagen(img_transmitida)
    img_farred_norm = normalizar_imagen(img_farred)

    farred_segmentada = np.ma.masked_where(
        ~mascara_final,
        img_farred_norm
    )

    plt.figure(figsize=(7, 7))

    plt.imshow(
        img_transmitida_norm,
        cmap="gray"
    )

    plt.imshow(
        farred_segmentada,
        cmap="inferno",
        alpha=alpha_farred
    )

    if mostrar_contorno and np.any(mascara_final):
        plt.contour(
            mascara_final,
            levels=[0.5],
            colors=color_contorno,
            linewidths=1.2
        )

    area_px = int(np.sum(mascara_final))

    plt.title(
        f"{replica} | {momento} | Pozo {pozo}\n"
        f"TRANSMITIDA + FARRED segmentada | Área = {area_px} px",
        fontsize=11,
        fontweight="bold"
    )

    plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"Imágenes exportadas/{replica}_{momento}_{pozo}_farred_transmitida_segmentada.png")
    plt.show()

In [ ]:
plot_overlay_farred_transmitida_mascara(
    mascaras_score=mascaras_score,
    replica="R1",
    momento="POST",
    pozo="3",
    base_path="Imágenes_experimento",
    alpha_farred=0.75,
    mostrar_contorno=True,
    color_contorno="cyan"
)

In [ ]:
for r in ["R1", "R2", "R3"]:
    for momento in ["PRE", "POST"]:
        for pozo in sorted(mascaras_score[r][momento].keys(), key=lambda x: int(x)):
            plot_overlay_farred_transmitida_mascara(
                mascaras_score=mascaras_score,
                replica=r,
                momento=momento,
                pozo=pozo,
                base_path="Imágenes_experimento",
                alpha_farred=0.75,
                mostrar_contorno=True,
                color_contorno="cyan"
            )

In [ ]:
def calcular_areas_y_generar_csv(
    todos,
    umbrales_otsu,
    elemento_estructurante,
    replicas=("R1", "R2", "R3"),
    momentos=("PRE", "POST"),
    nombre_csv="areas_medidas_por_replica_momento_pozo_categoria.csv",
    conectividad=2,
    margen_borde=0,
    area_minima=0
):
    """
    Recalcula la segmentación con score y genera un CSV con el área medida.

    No depende de df_metricas_score, resultados_score ni mascaras_score previos.
    """

    registros = []
    resultados_score = {}
    mascaras_score = {}

    for replica in replicas:
        resultados_score[replica] = {}
        mascaras_score[replica] = {}

        for momento in momentos:
            resultados_score[replica][momento] = {}
            mascaras_score[replica][momento] = {}

            if replica not in todos:
                continue

            if momento not in todos[replica]:
                continue

            pozos_ordenados = sorted(
                todos[replica][momento].keys(),
                key=lambda x: int(x)
            )

            for pozo in pozos_ordenados:

                res = analisis_morfologico_con_score(
                    todos=todos,
                    replica=replica,
                    momento=momento,
                    pozo=pozo,
                    umbrales_otsu=umbrales_otsu,
                    elemento_estructurante=elemento_estructurante,
                    conectividad=conectividad,
                    margen_borde=margen_borde,
                    area_minima=area_minima,
                    mostrar=False
                )

                mascara_final = res["mascara_score"]
                area_medida_px = int(np.sum(mascara_final))

                resultados_score[replica][momento][pozo] = res
                mascaras_score[replica][momento][pozo] = mascara_final

                registros.append({
                    "Replica": replica.replace("R", "Rep"),
                    "Momento": momento,
                    "Pozo": int(pozo),
                    "Categoria": "FARRED",
                    "Area_medida_px": area_medida_px
                })

    df_areas = pd.DataFrame(registros)

    df_areas["Replica_num"] = df_areas["Replica"].str.extract(r"Rep(\d+)").astype(int)

    df_areas = df_areas.sort_values(
        by=["Replica_num", "Momento", "Pozo", "Categoria"]
    )

    df_areas = df_areas[
        [
            "Replica",
            "Momento",
            "Pozo",
            "Categoria",
            "Area_medida_px"
        ]
    ]

    df_areas.to_csv(
        nombre_csv,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Archivo generado: {nombre_csv}")

    return df_areas, resultados_score, mascaras_score

In [ ]:
df_areas, resultados_score, mascaras_score = calcular_areas_y_generar_csv(
    todos=todos,
    umbrales_otsu=umbrales_otsu,
    elemento_estructurante=SE,
    replicas=("R1", "R2", "R3"),
    momentos=("PRE", "POST"),
    nombre_csv="areas_medidas_por_replica_momento_pozo_categoria.csv"
)

display(df_areas)

In [ ]:
"mascaras_score" in globals()